# Pronóstico de ventas por producto con PyTorch

Este notebook entrena una red neuronal para estimar las unidades vendidas de cada producto durante los próximos 7 días. Por defecto utiliza exclusivamente filas `synthetic`; nunca mezcla datos sintéticos y observados.

El CSV actual contiene 750 filas sintéticas (150 días para cada uno de 5 productos) y 5 filas observadas (una por producto). Las observaciones reales todavía no bastan para entrenar un modelo con datos reales.

La preparación toma ideas de exploración y escalado de **Un2_NoSupervisado_Parte1/Parte2**; el planteamiento de regresión, separación temporal y evaluación sigue **Un3**. El pronóstico es aprendizaje supervisado: una secuencia histórica de 28 días predice directamente los 7 siguientes. El origen del pronóstico toma solo la fecha más reciente del CSV; los valores de filas `observed` nunca entran al entrenamiento, validación, prueba ni a las características.

## 1. Entorno reproducible

Usa el kernel de Python 3.10.20 recomendado. PyTorch 2.5.1 funciona en ese entorno; este notebook también comprueba sus dependencias y fija las semillas de Python, NumPy y PyTorch. CPU es la opción predeterminada; la GPU se puede activar si CUDA está disponible. Si falta PyTorch en tu kernel, instálalo en ese entorno con `python -m pip install torch pandas numpy` y vuelve a seleccionar el kernel.

In [9]:
import importlib.util
import io
import random
import sys
from pathlib import Path

missing_packages = [name for name in ('torch', 'pandas', 'numpy') if importlib.util.find_spec(name) is None]
if missing_packages:
    raise ImportError(f'Faltan dependencias: {missing_packages}. Instálalas en el kernel con: python -m pip install torch pandas numpy')

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(1)

if sys.version_info[:2] != (3, 10):
    print(f'Aviso: el notebook se preparó para Python 3.10.20; este kernel usa {sys.version.split()[0]}.')

USE_GPU = False
DEVICE = torch.device('cuda' if USE_GPU and torch.cuda.is_available() else 'cpu')
if USE_GPU and DEVICE.type != 'cuda':
    print('CUDA no está disponible; se continuará en CPU.')
print(f'Python: {sys.version.split()[0]} | PyTorch: {torch.__version__} | Dispositivo: {DEVICE} | Semilla: {SEED}')

# Ruta editable para Jupyter local. En Colab puedes activar USE_COLAB_UPLOAD.
CSV_PATH = Path.home() / 'Downloads' / 'Ciencia de datos' / 'expo' / 'dataset' / 'inventory-history.csv'
USE_COLAB_UPLOAD = False

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if USE_COLAB_UPLOAD:
    if not IN_COLAB:
        raise RuntimeError('USE_COLAB_UPLOAD=True solo funciona dentro de Google Colab.')
    uploaded = colab_files.upload()
    if len(uploaded) != 1:
        raise ValueError('Sube exactamente un archivo CSV para continuar.')
    uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
    df_raw = pd.read_csv(io.BytesIO(uploaded_bytes))
    print(f'Archivo cargado desde Colab: {uploaded_name}')
else:
    if not CSV_PATH.is_file():
        raise FileNotFoundError(f'No se encontró el CSV: {CSV_PATH}. Edita CSV_PATH o activa USE_COLAB_UPLOAD en Colab.')
    df_raw = pd.read_csv(CSV_PATH)
    print(f'Archivo cargado: {CSV_PATH}')

print(f'Filas: {len(df_raw):,} | Columnas: {len(df_raw.columns)}')

Python: 3.10.20 | PyTorch: 2.5.1 | Dispositivo: cpu | Semilla: 42
Archivo cargado: C:\Users\Windows\Downloads\Ciencia de datos\expo\dataset\inventory-history.csv
Filas: 755 | Columnas: 10


## 2. Validación y diagnóstico del CSV

Se comprueba el esquema, se convierten fechas y números, y se informan duplicados, ausencias y posibles descuadres del inventario (`opening_stock + units_received - units_sold` frente a `stock_on_hand`). Las ausencias de `opening_stock` y `units_received` se permiten únicamente en filas observadas; esas variables no se usan como predictores.

In [10]:
REQUIRED_COLUMNS = ['date', 'product_id', 'product_name', 'category', 'price', 'opening_stock', 'stock_on_hand', 'units_sold', 'units_received', 'source']
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df_raw.columns))
if missing_columns:
    raise ValueError(f'Faltan columnas requeridas: {missing_columns}. Columnas recibidas: {list(df_raw.columns)}')

df = df_raw[REQUIRED_COLUMNS].copy()
for column in ('product_id', 'product_name', 'category', 'source'):
    df[column] = df[column].astype('string').str.strip()
df['source'] = df['source'].str.lower()
df['date'] = pd.to_datetime(df['date'], errors='coerce')
NUMERIC_COLUMNS = ['price', 'opening_stock', 'stock_on_hand', 'units_sold', 'units_received']
for column in NUMERIC_COLUMNS:
    df[column] = pd.to_numeric(df[column], errors='coerce')

print('Filas por fuente:')
print(df['source'].value_counts(dropna=False).to_string())
print('\nValores ausentes por columna:')
print(df.isna().sum().to_string())

duplicate_mask = df.duplicated(['product_id', 'date'], keep=False)
print(f'\nFilas con fecha duplicada para el mismo producto: {int(duplicate_mask.sum())}')
if duplicate_mask.any():
    print(df.loc[duplicate_mask, ['date', 'product_id', 'source']].head(10).to_string(index=False))

unknown_source = df['source'].isna() | ~df['source'].isin(['synthetic', 'observed'])
required_for_all = ['date', 'product_id', 'product_name', 'category', 'price', 'stock_on_hand', 'units_sold', 'source']
missing_required = df[required_for_all].isna().any(axis=1)
missing_synthetic_optional = df['source'].eq('synthetic') & df[['opening_stock', 'units_received']].isna().any(axis=1)
invalid_counts = pd.Series(False, index=df.index)
for column in ('opening_stock', 'stock_on_hand', 'units_sold', 'units_received'):
    values = df[column]
    invalid_counts |= values.notna() & ((values < 0) | (values % 1 != 0))
invalid_price = df['price'].notna() & ((df['price'] < 0) | ~np.isfinite(df['price']))
invalid_numeric = invalid_counts | invalid_price

synthetic_rows = df['source'].eq('synthetic')
has_stock_fields = df[['opening_stock', 'units_received', 'units_sold', 'stock_on_hand']].notna().all(axis=1)
expected_stock = df['opening_stock'] + df['units_received'] - df['units_sold']
stock_mismatch = synthetic_rows & has_stock_fields & expected_stock.ne(df['stock_on_hand'])
print(f'Inconsistencias de balance de inventario en filas sintéticas: {int(stock_mismatch.sum())}')
if stock_mismatch.any():
    print(df.loc[stock_mismatch, ['date', 'product_id', 'opening_stock', 'units_received', 'units_sold', 'stock_on_hand']].head(5).to_string(index=False))

invalid_rows = unknown_source | missing_required | missing_synthetic_optional | invalid_numeric
if invalid_rows.any():
    raise ValueError(f'Hay {int(invalid_rows.sum())} filas inválidas: revisa fechas/campos requeridos, source (synthetic u observed), números no negativos e inventario sintético completo. Las ausencias de opening_stock/units_received solo se permiten en observed.')

df = df.sort_values(['product_id', 'date']).reset_index(drop=True)
print('\nTipos convertidos:')
print(df.dtypes.to_string())

Filas por fuente:
source
synthetic    750
observed       5

Valores ausentes por columna:
date              0
product_id        0
product_name      0
category          0
price             0
opening_stock     5
stock_on_hand     0
units_sold        0
units_received    5
source            0

Filas con fecha duplicada para el mismo producto: 0
Inconsistencias de balance de inventario en filas sintéticas: 0

Tipos convertidos:
date              datetime64[ns]
product_id        string[python]
product_name      string[python]
category          string[python]
price                    float64
opening_stock            float64
stock_on_hand              int64
units_sold                 int64
units_received           float64
source            string[python]


## 3. Elección explícita de la fuente y partición temporal

Cambia `TRAIN_SOURCE` a `observed` solo si quieres intentar entrenar con snapshots reales; el notebook entonces selecciona únicamente esas filas y no las combina con las sintéticas. Para este modelo se requieren al menos 60 días continuos por producto. Si hay días faltantes o duplicados, el proceso se detiene en vez de inventar ventas cero. La última semana se reserva para prueba y la semana anterior para validación; nunca se mezclan ventanas futuras en el entrenamiento. Para fechar el pronóstico se usa solo la fecha más reciente del archivo, no sus ventas observadas: así se predicen días posteriores al snapshot sin incorporarlo al modelo.

In [11]:
TRAIN_SOURCE = 'synthetic'  # Cambiar explícitamente a 'observed' para probar solo snapshots reales.
INPUT_DAYS = 28
HORIZON = 7
MIN_DAYS_PER_PRODUCT = 60

if TRAIN_SOURCE not in ('synthetic', 'observed'):
    raise ValueError("TRAIN_SOURCE debe ser exactamente 'synthetic' u 'observed'.")
source_counts = df['source'].value_counts()
print(f"Filas observadas disponibles: {int(source_counts.get('observed', 0))}")
selected = df.loc[df['source'].eq(TRAIN_SOURCE)].copy()
print(f"Fuente elegida: {TRAIN_SOURCE} | Filas seleccionadas: {len(selected)}")
if selected.empty:
    raise ValueError(f'No hay filas de la fuente {TRAIN_SOURCE!r}.')

duplicate_selected = selected.duplicated(['product_id', 'date'], keep=False)
if duplicate_selected.any():
    raise ValueError(f'La fuente {TRAIN_SOURCE!r} contiene {int(duplicate_selected.sum())} filas con fechas duplicadas por producto; resuélvelas antes de entrenar.')

products = sorted(selected['product_id'].dropna().unique().tolist())
product_names = selected.groupby('product_id')['product_name'].last().to_dict()
series_by_product = []
for product_id in products:
    group = selected.loc[selected['product_id'].eq(product_id)].sort_values('date').reset_index(drop=True)
    if len(group) < MIN_DAYS_PER_PRODUCT:
        raise ValueError(f"Datos insuficientes para {product_names.get(product_id, product_id)!r}: {len(group)} días en {TRAIN_SOURCE!r}; se requieren al menos {MIN_DAYS_PER_PRODUCT}. Las filas observed actuales no bastan para entrenar con datos reales.")
    expected_dates = pd.date_range(group['date'].iloc[0], group['date'].iloc[-1], freq='D')
    if not group['date'].reset_index(drop=True).equals(pd.Series(expected_dates)):
        raise ValueError(f'Hay días faltantes en la serie de {product_names.get(product_id, product_id)!r}. Corrige el CSV; no se imputarán silenciosamente como ventas cero.')
    series_by_product.append((product_id, group))

print(f'Productos listos: {len(products)}')
print(selected.groupby('product_id').agg(dias=('date', 'size'), desde=('date', 'min'), hasta=('date', 'max')).to_string())

Filas observadas disponibles: 5
Fuente elegida: synthetic | Filas seleccionadas: 750
Productos listos: 5
               dias      desde      hasta
product_id                               
1               150 2026-06-26 2026-11-22
1781783885710   150 2026-06-26 2026-11-22
1782390715810   150 2026-06-26 2026-11-22
2               150 2026-06-26 2026-11-22
4               150 2026-06-26 2026-11-22


## 4. Ventanas, red neuronal y entrenamiento

Cada ejemplo contiene las ventas de los 28 días anteriores, el identificador del producto y el día de la semana de cada fecha que se pronostica. Se predicen los 7 valores a la vez con una red densa de regresión. Las ventas se transforman con `log1p` y se estandarizan usando únicamente el tramo de entrenamiento (práctica de escalado vista en Un2); la red se optimiza con Adam y error cuadrático medio. La validación temporal detiene el entrenamiento temprano.

In [12]:
def future_calendar_features(future_dates):
    weekdays = pd.DatetimeIndex(future_dates).dayofweek.to_numpy()
    angle = 2 * np.pi * weekdays / 7
    return np.column_stack((np.sin(angle), np.cos(angle))).astype(np.float32).reshape(-1)

train_log_values = []
for _, group in series_by_product:
    cutoff = len(group) - 2 * HORIZON
    train_log_values.extend(np.log1p(group['units_sold'].to_numpy(dtype=np.float32)[:cutoff]))
sales_mean = float(np.mean(train_log_values))
sales_scale = max(float(np.std(train_log_values)), 1e-6)

product_to_index = {product_id: index for index, product_id in enumerate(products)}
def make_features(history, product_id, future_dates):
    history_log = np.log1p(np.asarray(history, dtype=np.float32))
    history_scaled = (history_log - sales_mean) / sales_scale
    product_one_hot = np.zeros(len(products), dtype=np.float32)
    product_one_hot[product_to_index[product_id]] = 1.0
    return np.concatenate((history_scaled, product_one_hot, future_calendar_features(future_dates))).astype(np.float32)

X_train, y_train, X_valid, y_valid, X_test, y_test = [], [], [], [], [], []
test_actual, test_baseline, test_product_ids = [], [], []
for product_id, group in series_by_product:
    values = group['units_sold'].to_numpy(dtype=np.float32)
    dates = pd.DatetimeIndex(group['date'])
    n_days = len(group)
    validation_start = n_days - 2 * HORIZON
    test_start = n_days - HORIZON
    for target_start in range(INPUT_DAYS, validation_start - HORIZON + 1):
        history = values[target_start - INPUT_DAYS:target_start]
        target = values[target_start:target_start + HORIZON]
        X_train.append(make_features(history, product_id, dates[target_start:target_start + HORIZON]))
        y_train.append((np.log1p(target) - sales_mean) / sales_scale)

    history = values[validation_start - INPUT_DAYS:validation_start]
    target = values[validation_start:validation_start + HORIZON]
    X_valid.append(make_features(history, product_id, dates[validation_start:validation_start + HORIZON]))
    y_valid.append((np.log1p(target) - sales_mean) / sales_scale)

    history = values[test_start - INPUT_DAYS:test_start]
    target = values[test_start:test_start + HORIZON]
    X_test.append(make_features(history, product_id, dates[test_start:test_start + HORIZON]))
    y_test.append(target)
    test_actual.append(target)
    test_baseline.append(values[test_start - HORIZON:test_start])
    test_product_ids.append(product_id)

X_train = np.asarray(X_train, dtype=np.float32)
y_train = np.asarray(y_train, dtype=np.float32)
X_valid = torch.tensor(np.asarray(X_valid), dtype=torch.float32, device=DEVICE)
y_valid = torch.tensor(np.asarray(y_valid), dtype=torch.float32, device=DEVICE)
X_test = torch.tensor(np.asarray(X_test), dtype=torch.float32, device=DEVICE)
if len(X_train) == 0:
    raise ValueError('No se pudieron crear ventanas de entrenamiento; revisa la longitud de las series.')
print(f'Ejemplos de entrenamiento: {len(X_train)} | validación: {len(X_valid)} | prueba: {len(X_test)}')

def construir_modelo(input_size):
    return nn.Sequential(
        nn.Linear(input_size, 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, HORIZON),
    )

torch.manual_seed(SEED)
model = construir_modelo(X_train.shape[1]).to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, generator=loader_generator)

best_validation_loss = float('inf')
best_weights = None
epochs_without_improvement = 0
PATIENCE = 25
MAX_EPOCHS = 300
for epoch in range(MAX_EPOCHS):
    model.train()
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(DEVICE)
        batch_y = batch_y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        validation_loss = float(criterion(model(X_valid), y_valid).item())
    if validation_loss < best_validation_loss - 1e-6:
        best_validation_loss = validation_loss
        best_weights = {key: value.detach().clone() for key, value in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            break

if best_weights is None:
    raise RuntimeError('El entrenamiento no produjo un modelo válido.')
model.load_state_dict(best_weights)
print(f'Épocas ejecutadas: {epoch + 1} | pérdida de validación (MSE escalado): {best_validation_loss:.4f}')

def inverse_sales_scale(values):
    return np.maximum(0.0, np.expm1(np.asarray(values) * sales_scale + sales_mean))

model.eval()
with torch.no_grad():
    test_prediction_scaled = model(X_test).cpu().numpy()
test_prediction = inverse_sales_scale(test_prediction_scaled)
actual_matrix = np.asarray(test_actual)
baseline_matrix = np.asarray(test_baseline)
metric_rows = []
for index, product_id in enumerate(test_product_ids):
    actual = actual_matrix[index]
    predicted = test_prediction[index]
    baseline = baseline_matrix[index]
    metric_rows.append({
        'product_name': product_names[product_id],
        'MAE modelo': float(np.mean(np.abs(actual - predicted))),
        'RMSE modelo': float(np.sqrt(np.mean((actual - predicted) ** 2))),
        'MAE referencia semana previa': float(np.mean(np.abs(actual - baseline))),
    })
print('Evaluación en los últimos 7 días reservados (unidades):')
display(pd.DataFrame(metric_rows).rename(columns={'product_name': 'Producto'}).round(2))

forecast_origin = df['date'].max()
print(f'Fecha de referencia del CSV: {forecast_origin.date()} (solo se usa como fecha, no como dato de ventas).')
forecast_rows = []
for product_id, group in series_by_product:
    history = group['units_sold'].to_numpy(dtype=np.float32)[-INPUT_DAYS:]
    future_dates = pd.date_range(forecast_origin + pd.Timedelta(days=1), periods=HORIZON, freq='D')
    features = make_features(history, product_id, future_dates)
    with torch.no_grad():
        prediction_scaled = model(torch.tensor(features[None, :], dtype=torch.float32, device=DEVICE)).cpu().numpy()[0]
    prediction = inverse_sales_scale(prediction_scaled)
    for forecast_date, units in zip(future_dates, prediction):
        forecast_rows.append({
            'product_id': product_id,
            'product_name': product_names[product_id],
            'date': forecast_date.date(),
            'units_forecast': float(units),
        })

forecast_daily = pd.DataFrame(forecast_rows).sort_values(['product_name', 'date']).reset_index(drop=True)
forecast_totals = forecast_daily.groupby(['product_id', 'product_name'], as_index=False)['units_forecast'].sum()
print('Pronóstico diario para los próximos 7 días (unidades esperadas, decimales):')
display(forecast_daily[['product_name', 'date', 'units_forecast']].rename(columns={'product_name': 'Producto', 'date': 'Fecha', 'units_forecast': 'Unidades pronosticadas'}).round({'Unidades pronosticadas': 2}))
print('Total pronosticado por producto para los 7 días:')
display(forecast_totals.rename(columns={'product_id': 'ID del producto', 'product_name': 'Producto', 'units_forecast': 'Total pronosticado (7 días)'}).round({'Total pronosticado (7 días)': 2}))

Ejemplos de entrenamiento: 510 | validación: 5 | prueba: 5
Épocas ejecutadas: 31 | pérdida de validación (MSE escalado): 1.0179
Evaluación en los últimos 7 días reservados (unidades):


,Producto,MAE modelo,RMSE modelo,MAE referencia semana previa
0,Vino Aranjuez Terruño,1.00,1.14,1.00
1,Singani Casa Real Etiqueta Roja,1.07,1.31,1.29
2,nombre,0.77,0.97,1.00
3,Whisky Blue Label,2.54,2.78,2.43
4,Cerveza Paceña,0.90,1.11,2.71


Fecha de referencia del CSV: 2026-11-23 (solo se usa como fecha, no como dato de ventas).
Pronóstico diario para los próximos 7 días (unidades esperadas, decimales):


,Producto,Fecha,Unidades pronosticadas
0,Cerveza Paceña,2026-11-24,1.29
1,Cerveza Paceña,2026-11-25,2.01
2,Cerveza Paceña,2026-11-26,2.29
3,Cerveza Paceña,2026-11-27,2.15
4,Cerveza Paceña,2026-11-28,2.12
5,Cerveza Paceña,2026-11-29,1.25
6,Cerveza Paceña,2026-11-30,1.63
7,Singani Casa Real Etiqueta Roja,2026-11-24,1.76
8,Singani Casa Real Etiqueta Roja,2026-11-25,2.28
9,Singani Casa Real Etiqueta Roja,2026-11-26,2.88


Total pronosticado por producto para los 7 días:


,ID del producto,Producto,Total pronosticado (7 días)
0,1,Vino Aranjuez Terruño,4.91
1,1781783885710,Singani Casa Real Etiqueta Roja,15.59
2,1782390715810,nombre,10.08
3,2,Whisky Blue Label,12.87
4,4,Cerveza Paceña,12.74


## 5. Guardar y recuperar el modelo

Cambia `GUARDAR_MODELO` a `True` para crear un archivo `.joblib` en la ruta indicada. El archivo guarda los pesos de PyTorch junto con la arquitectura, el escalado y el orden de productos, necesarios para reconstruirlo. `cargar_modelo_joblib` permite recuperarlo sin volver a entrenar. Carga únicamente archivos `.joblib` de confianza.

In [14]:
GUARDAR_MODELO = True
RUTA_MODELO = Path.cwd() / 'pronostico_ventas.joblib'

def cargar_modelo_joblib(ruta=RUTA_MODELO, dispositivo=DEVICE):
    ruta = Path(ruta)
    if not ruta.is_file():
        raise FileNotFoundError(f'No existe el modelo guardado: {ruta}')
    if importlib.util.find_spec('joblib') is None:
        raise ImportError('Para cargar el modelo instala joblib con: python -m pip install joblib')
    import joblib

    artefacto = joblib.load(ruta)
    if artefacto.get('version_formato') != 1:
        raise ValueError('El archivo joblib no tiene una versión de formato compatible.')
    modelo_recuperado = construir_modelo(artefacto['tamano_entrada'])
    estado = {nombre: torch.from_numpy(peso) for nombre, peso in artefacto['pesos'].items()}
    modelo_recuperado.load_state_dict(estado)
    modelo_recuperado.to(dispositivo).eval()
    return modelo_recuperado, artefacto

if GUARDAR_MODELO:
    if importlib.util.find_spec('joblib') is None:
        raise ImportError('Para guardar el modelo instala joblib con: python -m pip install joblib')
    import joblib

    RUTA_MODELO.parent.mkdir(parents=True, exist_ok=True)
    artefacto = {
        'version_formato': 1,
        'pesos': {nombre: peso.detach().cpu().numpy() for nombre, peso in model.state_dict().items()},
        'tamano_entrada': int(X_train.shape[1]),
        'dias_entrada': INPUT_DAYS,
        'horizonte': HORIZON,
        'media_ventas_log': sales_mean,
        'escala_ventas_log': sales_scale,
        'productos': list(products),
        'nombres_productos': product_names,
        'indice_producto': product_to_index,
        'fuente_entrenamiento': TRAIN_SOURCE,
        'semilla': SEED,
        'version_pytorch': torch.__version__,
    }
    joblib.dump(artefacto, RUTA_MODELO)
    modelo_recuperado, _ = cargar_modelo_joblib(RUTA_MODELO)
    with torch.no_grad():
        torch.testing.assert_close(modelo_recuperado(X_test[:1]), model(X_test[:1]))
    print(f'Modelo guardado y verificado en: {RUTA_MODELO.resolve()}')

Modelo guardado y verificado en: C:\Users\Windows\Downloads\Ciencia de datos\expo\pronostico_ventas.joblib


## 6. Interpretación y límites

- `MAE` es el error absoluto medio en unidades; `RMSE` penaliza más los errores grandes. La columna de referencia compara con repetir la semana anterior. Son métricas de una sola semana de prueba, no una garantía de desempeño futuro.
- La tabla diaria presenta el producto, la fecha y las unidades pronosticadas; la tabla de totales suma los 7 días por producto. Los decimales son unidades esperadas de la regresión, no una promesa de cantidades enteras.
- El entrenamiento actual usa datos **sintéticos**, no ventas reales; el resultado sirve para practicar el flujo, no como pronóstico comercial validado. Hay solo 5 snapshots observados, uno por producto. Para entrenar con datos reales, recopila al menos 60 días diarios por producto, idealmente varios meses, y cambia explícitamente `TRAIN_SOURCE` a `observed`.
- La red solo usa ventas históricas, producto y día de la semana. No conoce precios futuros, promociones, cierres ni faltantes de inventario. No rellena fechas faltantes, porque asumir cero ventas cambiaría el significado de los datos.